[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/python_build123d_basics/blob/master/docs/tuto_colab_build/build123d_dxf_import_gc.ipynb)

# Importação de DXF: Layers, Planos, Extrusão, Sweep e Loft
## build123d para Arquitetos e Engenheiros
### Versão Google Colab

---

Neste notebook vamos importar perfis desenhados no AutoCAD (`.dxf`) e usá-los como base para modelagem 3D: **filtrar por layer**, **reposicionar em outros planos**, **extrudar**, usar um perfil como **caminho de sweep** para o outro, e fazer **loft** entre várias seções importadas.

---

## Instalação

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "cadquery-simpleviewer[build123d,interactive]"],
        check=True,
    )
    # build123d pulls in a newer ipython than Colab's kernel bootstrap
    # tolerates. Put Colab's version back on disk — do NOT restart the
    # runtime, the current kernel already has the working ipython loaded.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ipython==7.34.0", "--no-deps"],
        check=True,
    )

else:
    print("Not running in Colab, skipping package installation.")

## Importações

In [ ]:
from pathlib import Path

import build123d as b3d
import ezdxf
from cadquery_simpleviewer import show

---

## Arquivo de exemplo

Vamos reaproveitar o `perfil_autocad.dxf` usado no notebook de múltiplos pavimentos com perfil importado. Ele tem duas layers de interesse:

| Layer | Forma | Fechada? |
|-------|-------|----------|
| `profile_01` | polígono irregular de 6 lados | sim |
| `profile_02` | polígono irregular de 3 lados | sim |

No Colab o arquivo não existe no ambiente — baixamos direto do repositório do curso. Localmente (VS Code, Jupyter), o arquivo já está na mesma pasta do notebook.

In [ ]:
DXF_PATH = "perfil_autocad.dxf"

if IN_COLAB and not Path(DXF_PATH).exists():
    import urllib.request
    url = "https://raw.githubusercontent.com/255ribeiro/python_build123d_basics/master/docs/tuto_colab_build/perfil_autocad.dxf"
    urllib.request.urlretrieve(url, DXF_PATH)
    print("Arquivo baixado:", DXF_PATH)
else:
    print("Usando arquivo local:", DXF_PATH)

---

## Explorando as layers do DXF

Antes de importar, vale olhar o que existe dentro do arquivo. O **ezdxf** (a biblioteca que o próprio build123d usa por baixo dos panos para ler DXF) permite listar as layers e as entidades do modelspace sem passar pelo build123d.

In [ ]:
doc = ezdxf.readfile(DXF_PATH)
msp = doc.modelspace()

print("Layers do arquivo:")
for layer in doc.layers:
    print(" -", layer.dxf.name)

print()
print("Entidades no modelspace:")
contagem = {}
for entidade in msp:
    chave = (entidade.dxf.layer, entidade.dxftype())
    if chave in contagem:
        contagem[chave] += 1
    else:
        contagem[chave] = 1

for chave, quantidade in contagem.items():
    print(" -", chave, "->", quantidade)

---

## Importação "crua" — sem filtro de layer

`b3d.import_dxf()` lê **todas** as entidades do modelspace de uma vez, sem diferenciar de qual layer cada uma veio. O resultado é uma `ShapeList` de `Wire` — um por polilinha encontrada no arquivo.

> ⚠️ Diferente do `cq.importers.importDXF(..., include=[...])` do CadQuery, que já filtra a layer na própria importação, o build123d não tem esse parâmetro — qualquer filtro precisa ser feito **antes**, no próprio arquivo DXF ou com o ezdxf (próxima seção).

In [ ]:
perfis_todos = b3d.import_dxf(DXF_PATH)

print(f"Wires importados: {len(perfis_todos)}")
for perfil in perfis_todos:
    caixa = perfil.bounding_box()
    print(f" - fechado={perfil.is_closed}  bbox_x=({caixa.min.X:.2f}, {caixa.max.X:.2f})  bbox_y=({caixa.min.Y:.2f}, {caixa.max.Y:.2f})")

show(list(perfis_todos), visible_axes=None, z=0, plane_size=40)

---

## Filtrando por layer

Para importar **apenas uma layer**, usamos o ezdxf para copiar só as entidades desejadas para um arquivo DXF temporário, e deixamos o `import_dxf()` do build123d processar esse arquivo já filtrado:

In [ ]:
def import_dxf_layer(path, layer):
    """Importa apenas as entidades de uma layer específica de um DXF."""
    if not Path(path).exists():
        alternativo = Path("docs/tuto_colab_build") / path
        if alternativo.exists():
            path = str(alternativo)

    doc_origem = ezdxf.readfile(path)
    msp_origem = doc_origem.modelspace()

    doc_filtrado = ezdxf.new(dxfversion=doc_origem.dxfversion)
    msp_filtrado = doc_filtrado.modelspace()
    for entidade in msp_origem.query(f'*[layer=="{layer}"]'):
        msp_filtrado.add_foreign_entity(entidade)

    caminho_tmp = f"_layer_{layer}.dxf"
    doc_filtrado.saveas(caminho_tmp)
    return b3d.import_dxf(caminho_tmp)

In [ ]:
perfil_01 = import_dxf_layer(DXF_PATH, "profile_01")[0]
perfil_02 = import_dxf_layer(DXF_PATH, "profile_02")[0]

show(
    [perfil_01, perfil_02],
    names=["profile_01", "profile_02"],
    colors=["lightsteelblue", "lightsalmon"],
    visible_axes=None,
    z=0,
    plane_size=40
)

---

## Trocando o plano de importação

Um DXF é sempre um desenho **2D**: `import_dxf()` devolve as polilinhas achatadas no plano XY, com `Z = 0` — é a "prancha" onde o desenho foi feito no AutoCAD. Para usar esse perfil como um elemento **vertical** (um brise, uma viga, um painel de fachada), é preciso reposicioná-lo em outro `Plane` — a mesma técnica de multiplicação `plano * forma` já usada com sketches e wires nos notebooks anteriores.

| Plano | Efeito | Uso típico |
|-------|--------|------------|
| `Plane.XY` (padrão da importação) | Perfil permanece deitado | Planta baixa, pavimento — pronto para `extrude()` na vertical |
| `Plane.XZ` | Perfil vira vertical, de frente para -Y | Perfil de fachada, brise, viga |
| `Plane.YZ` | Perfil vira vertical, de frente para X | Perfil lateral |
| `Plane(origin=..., x_dir=..., z_dir=...)` | Orientação totalmente customizada | Perfil inclinado, telhado, rampa |

> 💡 `plano * forma` não modifica a forma original — devolve uma **cópia** já reposicionada. `perfil_01` continua deitado em XY depois da operação abaixo.

In [ ]:
face_01 = b3d.make_face(perfil_01)
print("Z do centro da face original (plano de importação):", face_01.center().Z)

face_xz = b3d.Plane.XZ * face_01
face_yz = b3d.Plane.YZ * face_01

print("Z do centro após Plane.XZ:", face_xz.center().Z)
print("Y do centro após Plane.YZ:", face_yz.center().Y)

show(
    [face_01, face_xz, face_yz],
    names=["Original (XY)", "Reposicionado em XZ", "Reposicionado em YZ"],
    colors=["lightsteelblue", "lightsalmon", "mediumpurple"],
    opacity=0.7,
    visible_axes=None,
    z=-10,
    plane_size=40
)

---

## Extrudando o perfil importado

Com o perfil no plano certo, `extrude()` funciona normalmente. Um perfil deitado em XY vira um volume **horizontal** (uma laje); o mesmo perfil reposicionado em XZ vira um painel **vertical**.

In [ ]:
laje = b3d.extrude(face_01, amount=1.0)

perfil_vertical = b3d.Plane.XZ * face_01
painel = b3d.extrude(perfil_vertical, amount=0.3)

show(
    [laje, painel],
    names=["Laje (extrude horizontal)", "Painel (extrude vertical)"],
    colors=["lightsteelblue", "lightsalmon"],
    visible_axes=None,
    z=-10,
    plane_size=40
)

---

## Sweep — varrendo um perfil ao longo do outro

Agora combinamos as duas layers do DXF: **`profile_01`** vira o perfil (a seção transversal que é arrastada) e **`profile_02`** vira o **caminho** (a trajetória do sweep) — como um corrimão ou uma moldura seguindo um traçado desenhado à mão livre no AutoCAD.

Duas adaptações são necessárias antes do `sweep()`:

1. **Centralizar e escalar o perfil** — `profile_01` tem ~25 unidades de largura, do tamanho do próprio caminho. Como seção transversal isso é enorme; centralizá-lo na origem e escalá-lo para baixo (`b3d.scale(..., by=0.05)`) dá um perfil pequeno o suficiente para "correr" ao longo do caminho.
2. **Abrir o caminho** — `profile_02` foi desenhado como um polígono **fechado** (3 arestas formando um laço). Um caminho de sweep normalmente é uma curva **aberta**; descartamos a última aresta (`perfil_02.edges()[:-1]`) para transformar o laço fechado em um traçado aberto de 2 segmentos, e reposicionamos esse caminho para o plano vertical `XZ` com a mesma técnica da seção anterior.

> ⚠️ **Cuidado com o parâmetro `transition`**: o caminho tem cantos vivos (não é uma curva suave). O valor padrão de `sweep()`, `Transition.TRANSFORMED`, pode fazer o perfil "estourar" para fora do caminho na dobra — o resultado abaixo mostra a comparação lado a lado. `Transition.RIGHT` corta o canto de forma limpa e é a escolha certa para caminhos poligonais com cantos vivos.

In [ ]:
centro = face_01.center()
perfil_secao = b3d.scale(face_01.translate((-centro.X, -centro.Y, -centro.Z)), by=0.05)

caminho_aberto = b3d.Wire(perfil_02.edges()[:-1])
caminho = b3d.Plane.XZ * caminho_aberto

sweep_padrao = b3d.sweep(perfil_secao, path=caminho)
sweep_corrigido = b3d.sweep(perfil_secao, path=caminho, transition=b3d.Transition.RIGHT)

show(
    [sweep_padrao, sweep_corrigido],
    names=["transition padrão (estoura no canto)", "transition=RIGHT (correto)"],
    colors=["lightsalmon", "lightsteelblue"],
    visible_axes=None,
    z=-10,
    plane_size=40
)

---

## Loft entre múltiplas seções importadas

`loft()` aceita qualquer sequência de `Face`/`Sketch` como seções, então podemos misturar as duas layers importadas com uma primitiva comum (`Circle`) para fazer a transição entre elas — a mesma lógica do loft de múltiplas seções já vista com formas primitivas, agora alimentada por perfis reais do DXF.

In [ ]:
centro_02 = b3d.make_face(perfil_02).center()
face_02_centrada = b3d.make_face(perfil_02).translate((-centro_02.X, -centro_02.Y, -centro_02.Z))

base  = face_01
meio  = b3d.Plane(origin=(0, 0, 8)) * b3d.scale(face_02_centrada, by=0.8)
topo  = b3d.Plane(origin=(0, 0, 14)) * b3d.Circle(4)

torre_loft = b3d.loft([base, meio, topo])

show(
    [torre_loft],
    names=["Loft: profile_01 → profile_02 → círculo"],
    colors=["lightsteelblue"],
    visible_axes=None,
    z=0,
    plane_size=40
)

---

## Exercício

Usando o mesmo `perfil_autocad.dxf` (ou um DXF próprio, exportado do AutoCAD/Rhino com pelo menos duas layers):

1. Liste as layers do arquivo com ezdxf antes de importar
2. Filtre e importe **cada layer separadamente** com `import_dxf_layer()`
3. Reposicione um dos perfis para um plano vertical (`Plane.XZ` ou `Plane.YZ`) e extrude-o
4. Use um dos perfis como **caminho** e o outro (centralizado e escalado) como **seção** em um `sweep()` — teste os dois valores de `transition` e compare
5. Faça um `loft()` com pelo menos três seções, misturando os perfis importados com uma primitiva (`Circle`, `RegularPolygon`, etc.)

Exiba cada etapa com `show()`.

In [ ]:
# Escreva seu código aqui

---

## Resumo

Neste notebook você aprendeu:

- Como listar layers e entidades de um DXF com **ezdxf**, antes mesmo de importar
- Como fazer uma importação "crua" com `b3d.import_dxf()`, que devolve uma `ShapeList` de `Wire` sem diferenciar layers
- Como **filtrar por layer**, copiando as entidades desejadas para um DXF temporário com ezdxf e importando só esse arquivo
- Que `import_dxf()` sempre devolve geometria **achatada no plano XY** (`Z = 0`) — e como **trocar o plano de importação** multiplicando por um `Plane` (`Plane.XZ`, `Plane.YZ`, ou um plano customizado)
- Como **extrudar** um perfil importado, tanto na horizontal (laje) quanto na vertical (painel), dependendo do plano escolhido
- Como usar **um perfil importado como caminho de sweep** para o outro — incluindo como abrir um laço fechado (`Wire(edges[:-1])`) e por que o parâmetro `transition=Transition.RIGHT` evita que o perfil "estoure" em cantos vivos
- Como fazer **loft** entre múltiplas seções, misturando perfis importados do DXF com primitivas do build123d

---
*build123d para Arquitetos e Engenheiros — Versão Google Colab*